# High-Throughput Data Cleaning & Parallelized PII Anonymization (v1)

Today's drill is to build a high-throughput text preprocessing and PII (Personally Identifiable Information) masking \
pipeline from scratch.

The pipeline will:
1. Clean noisy corporate chat text.

2. Remove HTML tags.

3. Normalize whitespace.

4. Remove non-printable characters.

5. Detect and mask:
   - Email Addresses → `[EMAIL]`

   - Phone Numbers → `[PHONE]`

   - Social Security Numbers → `[SSN]`

6. Process documents in parallel using Python's built-in `multiprocessing` module.

7. Benchmark execution time across configurable chunk sizes and worker counts.

## Requirements

### Regex Masking Engine

Implement compiled regular expressions for:
- Email Addresses

- Phone Numbers

- Social Security Numbers (SSN)

Replace matches with:

| PII Type | Replacement |
|-----------|-------------|
| Email | `[EMAIL]` |
| Phone | `[PHONE]` |
| SSN | `[SSN]` |

### Text Normalization Pipeline

Each document must be cleaned by:
1. Removing HTML tags

2. Normalizing whitespace

3. Removing non-printable characters

### Parallel Processing

Implement:

- Worker function

- Chunking logic

- Multiprocessing Pool execution

- Runtime benchmarking

### Expected Deliverable

A scalable preprocessing pipeline suitable for larger document collections.

### Imports

In [54]:
import re
import time
import multiprocess as mp

### Raw Corpus
Containing:
- HTML tags
- Emails
- Phone numbers
- SSNs
- Excess whitespace
- Non-printable characters

In [55]:
raw_corpus = [
    "  <p>Hey team,</p> can you review the log? User reported an issue. Email is john.doe123@company.co.uk and phone is +1-555-0199. ",
    "SYSTEM ALERT: Please purge SSN 999-12-3456 from the staging database immediately. No html here but   unnecessary   spaces. ",
    "<div>Manager note:</div> Call me back at (555) 014-3920 or ping admin@internal.net. Critical update needed.",
    "User_ID_404: Input string contained bad characters \x00\x1f and an unmasked SSN: 111223333. Fast track this."
]

# Compile Regex Patterns

Using compiled regex improves performance because patterns are compiled once and reused across all documents.

In [56]:
EMAIL_PATTERN = re.compile(r'\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b')

# Constrained to match standard formats without accidentally consuming raw 7-9 digit IDs
# Replaced \b with (?!\d) to prevent punctuation attachment failures
PHONE_PATTERN = re.compile(r'(?:\+?\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}(?!\d)')

SSN_PATTERN = re.compile(r'\b(?:\d{3}-\d{2}-\d{4}|\d{9})\b')

HTML_PATTERN = re.compile(r'<[^>]+>')

WHITESPACE_PATTERN = re.compile(r'\s+')

### Text Normalization
1. Remove HTML
2. Remove non-printable characters
3. Normalize whitespace
4. Trim leading/trailing spaces

In [57]:
def normalize_text(text):
    """ Normalize noisy text prior to PII masking """
    # Remove HTML tags
    text = HTML_PATTERN.sub(" ", text)

    # Remove non-printable characters
    text = ''.join(c for c in text if c.isprintable())

    # Normalize whitespace
    text = WHITESPACE_PATTERN.sub(" ", text)

    return text.strip()

### PII Masking Function
Apply masking rules after normalization.

Order matters:

1. Email
2. Phone
3. SSN

In [58]:
def mask_pii(text):
    """ Replace detected PII with category tokens. """
    text = EMAIL_PATTERN.sub("[EMAIL]", text)
    text = SSN_PATTERN.sub("[SSN]", text)
    text = PHONE_PATTERN.sub("[PHONE]", text)

    return text

### Single Document Processing Function

Combines:

- Normalization
- PII masking

into a single pipeline step.

In [59]:
def process_document(text):
    """ Combines cleaning and masking into a single pipeline step. """
    return mask_pii(normalize_text(text))

### Worker Function
Each worker receives a chunk of documents.

The worker:

1. Iterates through documents
2. Applies the cleaning pipeline
3. Returns cleaned results

In [60]:
def process_chunk(chunk):
    """ Worker entry point processing a structural slice of data. """
    return [process_document(doc) for doc in chunk]

### Chunking Utility
Simulates streaming ingestion by splitting the corpus into configurable batch sizes.

Benefits:

- Reduced IPC overhead
- Better CPU utilization
- Scalable architecture

In [61]:
def chunk_data(data, chunk_size):
    """ Generates sequential chunks from the source corpus. """
    for i in range(0, len(data), chunk_size):
        yield data[i:i + chunk_size]

### Parallel Processing Engine

Uses multiprocessing Pool to distribute chunks across CPU cores.

Configurable parameters:

- Number of processes
- Chunk size

In [62]:
def init_worker(context_dict):
    """Injects functions and regex targets directly into the worker's global scope."""
    globals().update(context_dict)

def parallel_clean_corpus(corpus, chunk_size=1000, num_processes=None):
    """ Processes corpus utilizing true multi-core pools. """
    chunks = list(chunk_data(corpus, chunk_size))

    start_time = time.perf_counter()

    # Capture all necessary setup logic from the current notebook state
    worker_context = {
        'EMAIL_PATTERN': EMAIL_PATTERN,
        'PHONE_PATTERN': PHONE_PATTERN,
        'SSN_PATTERN': SSN_PATTERN,
        'HTML_PATTERN': HTML_PATTERN,
        'WHITESPACE_PATTERN': WHITESPACE_PATTERN,
        'normalize_text': normalize_text,
        'mask_pii': mask_pii,
        'process_document': process_document
    }

    # Boot the pool with the initializer rule
    with mp.Pool(
        processes=num_processes, 
        initializer=init_worker, 
        initargs=(worker_context,)
    ) as pool:
        results = pool.map(process_chunk, chunks)

    elapsed = time.perf_counter() - start_time

    cleaned_documents = [doc for batch in results for doc in batch]

    return cleaned_documents, elapsed

### Run Pipeline
Execute the pipeline against the sample corpus.

In [63]:
""" Run Pipeline """
cleaned_corpus, runtime = parallel_clean_corpus(corpus=raw_corpus, chunk_size=2, num_processes=mp.cpu_count())

print("Cleaned Documents:\n")
for doc in cleaned_corpus:
    print(doc)
    
print(f"\nExecution Time: {runtime:.6f} seconds\n")

Cleaned Documents:

Hey team, can you review the log? User reported an issue. Email is [EMAIL] and phone is +1-555-0199.
SYSTEM ALERT: Please purge SSN [SSN] from the staging database immediately. No html here but unnecessary spaces.
Manager note: Call me back at [PHONE] or ping [EMAIL]. Critical update needed.
User_ID_404: Input string contained bad characters and an unmasked SSN: [SSN]. Fast track this.

Execution Time: 2.636456 seconds



# Benchmark Different Chunk Sizes
Larger datasets often benefit from tuning chunk size.

Too small:
- Excess IPC overhead

Too large:
- Poor load balancing

This benchmark helps identify the optimal configuration.

In [64]:
chunk_sizes = [1, 2, 4]

for size in chunk_sizes:

    _, runtime = parallel_clean_corpus(corpus=raw_corpus, chunk_size=size, num_processes=mp.cpu_count())

    print(
        f"Chunk Size = {size:<3} "
        f"Runtime = {runtime:.6f} sec"
    )

Chunk Size = 1   Runtime = 2.487643 sec
Chunk Size = 2   Runtime = 2.481285 sec
Chunk Size = 4   Runtime = 2.525920 sec


# Expected Output

```python
[
    "Hey team, can you review the log? User reported an issue. Email is [EMAIL] and phone is [PHONE].",
    "SYSTEM ALERT: Please purge SSN [SSN] from the staging database immediately. No html here but unnecessary spaces.",
    "Manager note: Call me back at [PHONE] or ping [EMAIL]. Critical update needed.",
    "User_ID_404: Input string contained bad characters and an unmasked SSN: [SSN]. Fast track this."
]